# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer: FAIR² Data Exploration with `mlcroissant`
This notebook guides users in loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. The dataset describes clinicopathological and molecular features (including MSI-H status and anatomical distribution) for 77 cancer survivors with second primary colorectal cancer.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields (columns), and their `@id`s in the dataset.

In [ ]:
# List all record sets and their IDs
print("Available record sets in dataset:")
record_sets = list(dataset.recordsets)
for record_set in record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '[no label]')}")

# For each record set, list the fields and their @id
for record_set in record_sets:
    print(f"\nFields/columns in record set {record_set['@id']}:")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  - {field['@id']}: {field.get('name', '[no label]')}")

## 3. Data Extraction
Load data from available record set(s) into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

If there are multiple record sets, you can extract each as a separate DataFrame for further analysis.

In [ ]:
#--- Step 1: Collect all record set IDs dynamically ---#
record_set_ids = [rs['@id'] for rs in dataset.recordsets]
dataframes = {}

#--- Step 2: Extract all records from each record set as a DataFrame ---#
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns for {record_set_id}:\n", dataframes[record_set_id].columns.tolist())
        print(f"First five rows from {record_set_id}:")
        display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

This section demonstrates common EDA steps using the main data record set (replace with the relevant `@id` from section 2/3).

* Filtering records (e.g. by age)
* Normalizing numeric columns
* Grouping (e.g. by sex or MSI-H status)

In [ ]:
# -- Identify main clinical record set (update the variable below based on available IDs from step 2/3) -- #
main_record_set_id = None
main_numeric_field_id = None
main_categorical_field_id = None
# Try to find a likely suitable clinical recordset:
for k, v in dataframes.items():
    if 'age' in [x.lower() for x in v.columns]:
        main_record_set_id = k
        # Try to find 'age' column exact field id
        for col in v.columns:
            if col.lower() == 'age':
                main_numeric_field_id = col
        break
# If not found, just use the first record set (fallback)
if main_record_set_id is None:
    main_record_set_id = record_set_ids[0]
    # Try to pick the first numeric column
    df = dataframes[main_record_set_id]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            main_numeric_field_id = col
            break
    # Use the first non-numeric for grouping as fallback
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            main_categorical_field_id = col
            break

print(f"Selected record set: {main_record_set_id}")
print(f"Numeric field (for filtering/normalization): {main_numeric_field_id}")
if main_categorical_field_id is None:
    # Try to fall back to any string/categorical field
    for col in dataframes[main_record_set_id].columns:
        if col != main_numeric_field_id:
            main_categorical_field_id = col
            break
print(f"Group field: {main_categorical_field_id}")

#-- Filter records: e.g. only Age > 60 --#
threshold = 60
df = dataframes[main_record_set_id]
if main_numeric_field_id is not None:
    filtered_df = df[df[main_numeric_field_id] > threshold]
    print(f"Filtered {len(filtered_df)} records where {main_numeric_field_id} > {threshold}")
    display(filtered_df.head())
    # Normalization (z-score)
    filtered_df[f"{main_numeric_field_id}_normalized"] = (
        filtered_df[main_numeric_field_id] - filtered_df[main_numeric_field_id].mean()
    ) / filtered_df[main_numeric_field_id].std()
    print(f"Normalized {main_numeric_field_id}:")
    display(filtered_df[[main_numeric_field_id, f"{main_numeric_field_id}_normalized"]].head())
else:
    print("No numeric field found for analysis.")
    filtered_df = None

#-- Grouped statistics --#
if filtered_df is not None and main_categorical_field_id is not None and main_categorical_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(main_categorical_field_id)[main_numeric_field_id].mean().reset_index()
    print(f"Mean {main_numeric_field_id} by {main_categorical_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize selected data distributions or relationships between fields in the dataset, such as age distribution or the proportion of MSI-H phenotype.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field (e.g. Age)
if main_numeric_field_id is not None and main_record_set_id in dataframes:
    plt.figure(figsize=(8, 4))
    df = dataframes[main_record_set_id]
    sns.histplot(df[main_numeric_field_id].dropna(), bins=10, kde=True)
    plt.xlabel(main_numeric_field_id)
    plt.title(f"Distribution of {main_numeric_field_id}")
    plt.show()

# Barplot of group field average, if available
if filtered_df is not None and main_categorical_field_id is not None and main_categorical_field_id in filtered_df.columns:
    plt.figure(figsize=(10,4))
    sns.barplot(x=main_categorical_field_id, y=main_numeric_field_id, data=filtered_df)
    plt.title(f"Average {main_numeric_field_id} by {main_categorical_field_id} (Filtered)")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

* The FAIR² dataset provides comprehensive clinicopathological and molecular (MSI-H) data for cancer survivors with second primary colorectal cancer.
* You can efficiently explore, filter, process, and visualize this data using `mlcroissant` while referencing entities by their `@id`s.
* For more details, see the Croissant schema or refer to [mlcroissant documentation](https://mlcroissant.readthedocs.io/).
